In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F

catalog_name = 'ecommerce'


In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show(10)

+----------+-----------+-------------+--------------------+--------------------+
|brand_code| brand_name|category_code|         source_file|         ingested_at|
+----------+-----------+-------------+--------------------+--------------------+
|      ACME|   AcmeTech|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      NOVW|  NovaWave |           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      ZNTH|     Zenith|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      BYTM|    ByteMax|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      ECOT|    EcoTone|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      SKYL|    SkyLink|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|     VOLT@|   VoltEdge|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      PHTX|   Photonix|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      URTL| UrbanTrail|          APP|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      COTC| CottonClub|    

In [0]:
#remove spaces
df_silver = df_bronze.withColumn('brand_name', F.trim(F.col('brand_name')))
df_silver.show(10)

+----------+----------+-------------+--------------------+--------------------+
|brand_code|brand_name|category_code|         source_file|         ingested_at|
+----------+----------+-------------+--------------------+--------------------+
|      ACME|  AcmeTech|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      NOVW|  NovaWave|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      ZNTH|    Zenith|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      BYTM|   ByteMax|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      ECOT|   EcoTone|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      SKYL|   SkyLink|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|     VOLT@|  VoltEdge|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      PHTX|  Photonix|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      URTL|UrbanTrail|          APP|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      COTC|CottonClub|          APP|dbf

In [0]:
#remove symbols like @, # etc
df_silver = df_silver.withColumn("brand_code", F.regexp_replace(F.col("brand_code"), r'[^A-Za-z0-9]', ''))
df_silver.show(10)

+----------+----------+-------------+--------------------+--------------------+
|brand_code|brand_name|category_code|         source_file|         ingested_at|
+----------+----------+-------------+--------------------+--------------------+
|      ACME|  AcmeTech|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      NOVW|  NovaWave|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      ZNTH|    Zenith|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      BYTM|   ByteMax|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      ECOT|   EcoTone|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      SKYL|   SkyLink|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      VOLT|  VoltEdge|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      PHTX|  Photonix|           CE|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      URTL|UrbanTrail|          APP|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|      COTC|CottonClub|          APP|dbf

In [0]:
#see the distinct/unique values
df_silver.select("category_code").distinct().show()


+-------------+
|category_code|
+-------------+
|           CE|
|          APP|
|          HNK|
|          BPC|
|        BOOKS|
|          BKS|
|      GROCERY|
|         GRCY|
|          TOY|
|         TOYS|
|          SPT|
+-------------+



as we can see that Books and bks can be same, toy and toys are same, grocery and grcy is same.

this is called as anomalies

In [0]:
# Anomalies Dict

anomalies = {
      "BOOKS" : "BKS" , 
      "GROCERY" : "GRCY",
      "TOYS" : "TOY"
}

df_silver= df_silver.replace(anomalies, subset="category_code")

df_silver.select("category_code").distinct().show()


+-------------+
|category_code|
+-------------+
|           CE|
|          APP|
|          HNK|
|          BPC|
|          BKS|
|         GRCY|
|          TOY|
|          SPT|
+-------------+



In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.silver.slv_brands")

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")

df_bronze.show()


+-------------+--------------------+--------------------+--------------------+
|category_code|       category_name|         source_file|         ingested_at|
+-------------+--------------------+--------------------+--------------------+
|           ce|         Electronics|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|          app|             Apparel|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|          hnk|      Home & Kitchen|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|          bpc|Beauty & Personal...|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|          bks|               Books|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|         grcy|             Grocery|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|          toy|        Toys & Games|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|          spt|   Sports & Outdoors|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|          app|             Apparel|dbfs:/Volumes/eco...|2026-06-02 20:28:...|
|         grcy|             Grocery|dbfs:/Volumes/ec

In [0]:
df_duplicates = df_bronze.groupBy("category_code").count().filter(F.col("count") > 1)
df_duplicates.show()



+-------------+-----+
|category_code|count|
+-------------+-----+
|          app|    2|
|         grcy|    2|
+-------------+-----+



In [0]:
df_silver = df_bronze.dropDuplicates(['category_code'])
display(df_silver)

category_code,category_name,source_file,ingested_at
ce,Electronics,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z
app,Apparel,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z
hnk,Home & Kitchen,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z
bpc,Beauty & Personal Care,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z
bks,Books,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z
grcy,Grocery,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z
toy,Toys & Games,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z
spt,Sports & Outdoors,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z


In [0]:
df_silver = df_silver.withColumn("category_code", F.upper(F.col("category_code")))
display(df_silver)

category_code,category_name,source_file,ingested_at
CE,Electronics,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z
APP,Apparel,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z
HNK,Home & Kitchen,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z
BPC,Beauty & Personal Care,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z
BKS,Books,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z
GRCY,Grocery,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z
TOY,Toys & Games,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z
SPT,Sports & Outdoors,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-06-02T20:28:31.271Z


In [0]:
df_silver.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(f"{catalog_name}.silver.slv_category")
